# Day 05 — NumPy: Arrays, Indexing & Math Operations



In [1]:
import numpy as np

print("NumPy version:", np.__version__)

NumPy version: 2.5.3


### Why NumPy instead of lists?
- **Faster:** operations run in compiled C code, applied to the whole array at once (*vectorization*).
- **Less memory:** all elements share one data type, stored contiguously.
- **Cleaner code:** `a + b` instead of loops.

In [2]:
import time

n = 1_000_000
py_list = list(range(n))
np_arr = np.arange(n)

t0 = time.perf_counter()
doubled_list = [x * 2 for x in py_list]
t_list = time.perf_counter() - t0

t0 = time.perf_counter()
doubled_arr = np_arr * 2
t_np = time.perf_counter() - t0

print(f"list comprehension: {t_list:.4f}s")
print(f"numpy vectorized  : {t_np:.4f}s")
print(f"numpy is roughly {t_list / t_np:.0f}x faster here")

list comprehension: 0.0558s
numpy vectorized  : 0.0062s
numpy is roughly 9x faster here


---
## 1. Creating Arrays

### 1.1 From Python lists

In [3]:
a = np.array([1, 2, 3, 4, 5])             # 1-D
m = np.array([[1, 2, 3], [4, 5, 6]])      # 2-D (2 rows, 3 columns)
t = np.array([[[1, 2], [3, 4]],
              [[5, 6], [7, 8]]])           # 3-D

print(a)
print(m)
print(t)

[1 2 3 4 5]
[[1 2 3]
 [4 5 6]]
[[[1 2]
  [3 4]]

 [[5 6]
  [7 8]]]


### 1.2 Built-in constructors

In [4]:
print("zeros:\n", np.zeros((2, 3)))
print("ones:\n", np.ones((2, 2)))
print("full:\n", np.full((2, 3), 7))
print("identity:\n", np.eye(3))
print("empty-like zeros:", np.zeros_like(m))

zeros:
 [[0. 0. 0.]
 [0. 0. 0.]]
ones:
 [[1. 1.]
 [1. 1.]]
full:
 [[7 7 7]
 [7 7 7]]
identity:
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
empty-like zeros: [[0 0 0]
 [0 0 0]]


### 1.3 Ranges and evenly spaced values

In [5]:
print(np.arange(10))              # 0..9
print(np.arange(2, 20, 4))        # start, stop, step
print(np.linspace(0, 1, 5))       # 5 evenly spaced points, endpoints included
print(np.logspace(0, 3, 4))       # 10^0 .. 10^3

[0 1 2 3 4 5 6 7 8 9]
[ 2  6 10 14 18]
[0.   0.25 0.5  0.75 1.  ]
[   1.   10.  100. 1000.]


### 1.4 Random arrays

In [6]:
rng = np.random.default_rng(seed=42)      # modern, reproducible generator

print(rng.random(3))                       # uniform [0, 1)
print(rng.integers(1, 10, size=(2, 4)))    # random ints in [1, 10)
print(rng.normal(loc=0, scale=1, size=3))  # normal distribution
print(rng.choice(["a", "b", "c"], size=5))

[0.77395605 0.43887844 0.85859792]
[[1 7 2 1]
 [5 9 7 7]]
[-0.31624259 -0.01680116 -0.85304393]
['b' 'b' 'a' 'c' 'c']


### 1.5 Data types

In [7]:
ints = np.array([1, 2, 3])
floats = np.array([1.0, 2.5, 3.0])
mixed = np.array([1, 2.5, 3])             # upcast to float
explicit = np.array([1, 2, 3], dtype=np.float32)

print(ints.dtype, floats.dtype, mixed.dtype, explicit.dtype)

# Convert with astype (returns a new array)
print(floats.astype(int))                  # truncates, doesn't round
print(np.array([0, 1, 2]).astype(bool))
print(np.array(["1", "2", "3"]).astype(int))

int64 float64 float64 float32
[1 2 3]
[False  True  True]
[1 2 3]


---
## 2. Array Attributes & Reshaping

### 2.1 Inspecting an array

In [8]:
arr = np.arange(12).reshape(3, 4)
print(arr)

print("ndim :", arr.ndim)       # number of dimensions
print("shape:", arr.shape)      # (rows, columns)
print("size :", arr.size)       # total elements
print("dtype:", arr.dtype)
print("itemsize:", arr.itemsize, "bytes per element")
print("nbytes  :", arr.nbytes, "bytes total")

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
ndim : 2
shape: (3, 4)
size : 12
dtype: int64
itemsize: 8 bytes per element
nbytes  : 96 bytes total


### 2.2 Reshaping

In [9]:
a = np.arange(12)

print(a.reshape(3, 4))
print(a.reshape(2, 6))
print(a.reshape(2, -1))        # -1 means "figure it out"
print(a.reshape(2, 2, 3).shape)

m = a.reshape(3, 4)
print("transpose shape:", m.T.shape)
print("flatten:", m.flatten())
print("ravel  :", m.ravel())    # view when possible, flatten always copies

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
[[ 0  1  2  3  4  5]
 [ 6  7  8  9 10 11]]
[[ 0  1  2  3  4  5]
 [ 6  7  8  9 10 11]]
(2, 2, 3)
transpose shape: (4, 3)
flatten: [ 0  1  2  3  4  5  6  7  8  9 10 11]
ravel  : [ 0  1  2  3  4  5  6  7  8  9 10 11]


### 2.3 Adding dimensions

In [10]:
v = np.array([1, 2, 3])
print(v.shape)

col = v[:, np.newaxis]         # shape (3, 1)
row = v[np.newaxis, :]         # shape (1, 3)
print(col.shape, row.shape)

print(np.expand_dims(v, axis=0).shape)
print(np.squeeze(col).shape)   # removes length-1 dimensions

(3,)
(3, 1) (1, 3)
(1, 3)
(3,)


### 2.4 Joining and splitting

In [11]:
a = np.array([[1, 2], [3, 4]])
b = np.array([[5, 6], [7, 8]])

print("vstack:\n", np.vstack([a, b]))
print("hstack:\n", np.hstack([a, b]))
print("concatenate axis=1:\n", np.concatenate([a, b], axis=1))

big = np.arange(12).reshape(3, 4)
left, right = np.hsplit(big, 2)
print("left:\n", left)
print("right:\n", right)
print([p.shape for p in np.vsplit(big, 3)])

vstack:
 [[1 2]
 [3 4]
 [5 6]
 [7 8]]
hstack:
 [[1 2 5 6]
 [3 4 7 8]]
concatenate axis=1:
 [[1 2 5 6]
 [3 4 7 8]]
left:
 [[0 1]
 [4 5]
 [8 9]]
right:
 [[ 2  3]
 [ 6  7]
 [10 11]]
[(1, 4), (1, 4), (1, 4)]


---
## 3. Indexing & Slicing

### 3.1 1-D indexing and slicing

In [13]:
a = np.array([10, 20, 30, 40, 50, 60])

print(a[0], a[-1])
print(a[1:4])
print(a[:3])
print(a[::2])
print(a[::-1])

10 60
[20 30 40]
[10 20 30]
[10 30 50]
[60 50 40 30 20 10]


### 3.2 2-D indexing

In [14]:
m = np.array([[1,  2,  3,  4],
              [5,  6,  7,  8],
              [9, 10, 11, 12]])

print(m[1, 2])        # row 1, column 2  -> 7
print(m[1])           # whole row
print(m[:, 0])        # whole column
print(m[0:2, 1:3])    # sub-block
print(m[-1, -1])      # last element
print(m[::2, ::2])    # every other row & column

7
[5 6 7 8]
[1 5 9]
[[2 3]
 [6 7]]
12
[[ 1  3]
 [ 9 11]]


### 3.3 Slices are *views*, not copies

In [15]:
a = np.arange(6)
s = a[1:4]
s[0] = 999            # modifies the original too!
print("a:", a)

b = np.arange(6)
c = b[1:4].copy()     # an independent copy
c[0] = 999
print("b:", b)
print("shares memory:", np.shares_memory(a, s), np.shares_memory(b, c))

a: [  0 999   2   3   4   5]
b: [0 1 2 3 4 5]
shares memory: True False


### 3.4 Boolean (mask) indexing

In [16]:
scores = np.array([55, 82, 91, 47, 68, 99, 73])

mask = scores >= 70
print(mask)
print(scores[mask])
print(scores[scores < 60])

# Combine conditions with & | ~  (parentheses are required)
print(scores[(scores > 60) & (scores < 90)])
print(scores[~(scores > 60)])

# Modify by condition
curved = scores.copy()
curved[curved < 60] = 60
print(curved)

[False  True  True False False  True  True]
[82 91 99 73]
[55 47]
[82 68 73]
[55 47]
[60 82 91 60 68 99 73]


### 3.5 Fancy indexing (index with arrays)

In [17]:
a = np.array([100, 200, 300, 400, 500])

print(a[[0, 2, 4]])
print(a[[-1, -2]])

m = np.arange(12).reshape(3, 4)
print(m[[0, 2]])                  # rows 0 and 2
print(m[[0, 1, 2], [1, 2, 3]])    # elements (0,1), (1,2), (2,3)
print(m[:, [0, 3]])               # columns 0 and 3

[100 300 500]
[500 400]
[[ 0  1  2  3]
 [ 8  9 10 11]]
[ 1  6 11]
[[ 0  3]
 [ 4  7]
 [ 8 11]]


### 3.6 `where`, `argmax`, `argsort`

In [18]:
a = np.array([4, 9, 1, 7, 3, 9])

print(np.where(a > 5))                   # indices where condition is true
print(np.where(a > 5, "big", "small"))   # choose values elementwise
print("argmax:", a.argmax(), "| argmin:", a.argmin())
print("argsort:", np.argsort(a))
print("sorted :", np.sort(a))
print("top 3  :", a[np.argsort(a)[-3:]])

(array([1, 3, 5]),)
['small' 'big' 'small' 'big' 'small' 'big']


argmax: 1 | argmin: 2
argsort: [2 4 0 3 1 5]
sorted : [1 3 4 7 9 9]
top 3  : [7 9 9]


---
## 4. Math Operations

### 4.1 Elementwise arithmetic

In [20]:
a = np.array([1, 2, 3, 4])
b = np.array([10, 20, 30, 40])

print("a + b :", a + b)
print("b - a :", b - a)
print("a * b :", a * b)       # elementwise, NOT matrix multiplication
print("b / a :", b / a)
print("b // 3:", b // 3)
print("b % 7 :", b % 7)
print("a ** 2:", a ** 2)

# Scalars apply to every element
print("a * 10 + 1:", a * 10 + 1)

a + b : [11 22 33 44]
b - a : [ 9 18 27 36]
a * b : [ 10  40  90 160]
b / a : [10. 10. 10. 10.]
b // 3: [ 3  6 10 13]
b % 7 : [3 6 2 5]
a ** 2: [ 1  4  9 16]
a * 10 + 1: [11 21 31 41]


### 4.2 Universal functions (ufuncs)

In [21]:
x = np.array([0, np.pi / 2, np.pi])

print(np.sin(x).round(3))
print(np.cos(x).round(3))
print(np.exp([0, 1, 2]).round(3))
print(np.log([1, np.e, 10]).round(3))
print(np.sqrt([1, 4, 9, 16]))
print(np.abs([-3, 2, -1]))
print(np.round([1.234, 5.678], 1), np.floor([1.7, -1.7]), np.ceil([1.2, -1.2]))

[0. 1. 0.]
[ 1.  0. -1.]
[1.    2.718 7.389]
[0.    1.    2.303]
[1. 2. 3. 4.]
[3 2 1]
[1.2 5.7] [ 1. -2.] [ 2. -1.]


### 4.3 Comparison and logical operations

In [22]:
a = np.array([1, 5, 3, 8])
b = np.array([2, 5, 1, 9])

print(a == b)
print(a > b)
print(np.array_equal(a, a.copy()))
print(np.allclose([0.1 + 0.2], [0.3]))
print(np.logical_and(a > 2, b > 2))
print("any > 7:", (a > 7).any(), "| all > 0:", (a > 0).all())

[False  True False False]
[False False  True False]
True
True
[False  True False  True]
any > 7: True | all > 0: True


### 4.4 Aggregations

In [23]:
data = np.array([[3, 7, 1],
                 [9, 2, 6],
                 [4, 8, 5]])

print("sum  :", data.sum())
print("mean :", data.mean())
print("std  :", data.std().round(3))
print("min/max:", data.min(), data.max())
print("median:", np.median(data))
print("product of [1,2,3,4]:", np.prod([1, 2, 3, 4]))
print("cumulative sum:", np.cumsum([1, 2, 3, 4]))

sum  : 45
mean : 5.0
std  : 2.582
min/max: 1 9
median: 5.0
product of [1,2,3,4]: 24
cumulative sum: [ 1  3  6 10]


#### Using the `axis` argument
- `axis=0` → collapse the **rows** (result has one value per **column**)
- `axis=1` → collapse the **columns** (result has one value per **row**)

In [24]:
print(data)
print("column sums (axis=0):", data.sum(axis=0))
print("row sums    (axis=1):", data.sum(axis=1))
print("column means        :", data.mean(axis=0).round(2))
print("max of each row     :", data.max(axis=1))
print("keepdims shape      :", data.sum(axis=1, keepdims=True).shape)

[[3 7 1]
 [9 2 6]
 [4 8 5]]
column sums (axis=0): [16 17 12]
row sums    (axis=1): [11 17 17]
column means        : [5.33 5.67 4.  ]
max of each row     : [7 9 8]
keepdims shape      : (3, 1)


### 4.5 Broadcasting

In [25]:
# Rule: align shapes from the right; dimensions match if equal or one of them is 1.

m = np.array([[1, 2, 3],
              [4, 5, 6]])         # shape (2, 3)

row = np.array([10, 20, 30])      # shape (3,)  -> stretched down the rows
col = np.array([[100], [200]])    # shape (2, 1) -> stretched across the columns

print(m + row)
print(m + col)
print(m * 2)

# Outer product through broadcasting
a = np.array([1, 2, 3])
print(a[:, None] * a[None, :])

[[11 22 33]
 [14 25 36]]
[[101 102 103]
 [204 205 206]]
[[ 2  4  6]
 [ 8 10 12]]
[[1 2 3]
 [2 4 6]
 [3 6 9]]


#### Broadcasting in practice: standardizing columns

In [26]:
rng = np.random.default_rng(0)
X = rng.integers(0, 100, size=(5, 3)).astype(float)

mean = X.mean(axis=0)             # shape (3,)
std = X.std(axis=0)

Z = (X - mean) / std              # broadcasting does the work
print(X)
print("column means after:", Z.mean(axis=0).round(6))
print("column stds after :", Z.std(axis=0).round(6))

[[85. 63. 51.]
 [26. 30.  4.]
 [ 7.  1. 17.]
 [81. 64. 91.]
 [50. 60. 97.]]
column means after: [ 0. -0.  0.]
column stds after : [1. 1. 1.]


### 4.6 Linear algebra basics

In [27]:
A = np.array([[1, 2],
              [3, 4]])
B = np.array([[5, 6],
              [7, 8]])

print("elementwise A * B:\n", A * B)
print("matrix product A @ B:\n", A @ B)
print("transpose:\n", A.T)
print("determinant:", round(np.linalg.det(A), 3))
print("inverse:\n", np.linalg.inv(A).round(3))

# Solve the system  2x + y = 5,  x + 3y = 10
coeffs = np.array([[2, 1], [1, 3]])
const = np.array([5, 10])
print("solution [x, y]:", np.linalg.solve(coeffs, const))

v = np.array([3, 4])
print("dot:", np.dot(v, v), "| norm:", np.linalg.norm(v))

elementwise A * B:
 [[ 5 12]
 [21 32]]
matrix product A @ B:
 [[19 22]
 [43 50]]
transpose:
 [[1 3]
 [2 4]]
determinant: -2.0
inverse:
 [[-2.   1. ]
 [ 1.5 -0.5]]
solution [x, y]: [1. 3.]
dot: 25 | norm: 5.0


### 4.7 Handling missing values (`NaN`) and special values

In [28]:
a = np.array([1.0, np.nan, 3.0, np.nan, 5.0])

print(a.sum())                  # nan spreads
print(np.nansum(a), np.nanmean(a))
print(np.isnan(a))
print(a[~np.isnan(a)])

# Division by zero gives inf (plus a warning, silenced here)
with np.errstate(divide="ignore"):
    print(1 / np.array([0.0, 2.0]))

nan
9.0 3.0
[False  True False  True False]
[1. 3. 5.]
[inf 0.5]


---
## 5. Mini Project: Student Grade Analysis

Everything together: creating, indexing, masking, and aggregating.

In [30]:
rng = np.random.default_rng(7)
students = np.array(["Ana", "Ben", "Cy", "Dia", "Eli", "Fay"])
subjects = ["Math", "Physics", "English"]

grades = rng.integers(50, 100, size=(6, 3))
print(subjects)
print(grades)

print("\nAverage per subject:", dict(zip(subjects, grades.mean(axis=0).round(1))))

student_avg = grades.mean(axis=1)
best = student_avg.argmax()
print(f"Top student: {students[best]} ({student_avg[best]:.1f})")

print("Students above 80 avg:", students[student_avg > 80])

# Students who scored below 60 in at least one subject
needs_help = (grades < 60).any(axis=1)
print("Needs support:", students[needs_help])

# Ranking
order = np.argsort(-student_avg)
for rank, i in enumerate(order, start=1):
    print(f"{rank}. {students[i]:<4} {student_avg[i]:.1f}")

# Curve: +5 points, capped at 100
curved = np.clip(grades + 5, 0, 100)
print("\nAfter curve:\n", curved)

['Math', 'Physics', 'English']
[[97 81 84]
 [94 78 88]
 [91 61 52]
 [65 64 93]
 [95 50 74]
 [91 56 89]]

Average per subject: {'Math': np.float64(88.8), 'Physics': np.float64(65.0), 'English': np.float64(80.0)}
Top student: Ana (87.3)
Students above 80 avg: ['Ana' 'Ben']
Needs support: ['Cy' 'Eli' 'Fay']


1. Ana  87.3
2. Ben  86.7
3. Fay  78.7
4. Dia  74.0
5. Eli  73.0
6. Cy   68.0

After curve:
 [[100  86  89]
 [ 99  83  93]
 [ 96  66  57]
 [ 70  69  98]
 [100  55  79]
 [ 96  61  94]]




### Gotchas to remember
- `*` is elementwise; use `@` for matrix multiplication.
- Slices share memory with the original; call `.copy()` if you need independence.
- Use `&`, `|`, `~` (with parentheses) for conditions, not `and`, `or`, `not`.
- `axis=0` works down the rows (per column); `axis=1` works across columns (per row).

